# 04 — Gemma4-31B via Ollama Cloud API (accuracy-only, retrieval + summarization)

No download, no local server, no GPU needed. Calls `https://ollama.com/api/chat`
(the same `/api/chat` shape as a local server) with model **`gemma4:31b`**
(dense 31B, 256K ctx, Text+Image in — verified visible via `/api/tags` 2026-10-03).

**Rules for this arm:** accuracy/quality metrics only (hit rate, F1, ROUGE).
Latency/VRAM columns are `null` by design — never compare them with any other
run. Key comes from Kaggle Secret `OLLAMA_API_KEY` (ollama.com → Settings →
API keys), never hardcoded. Fairness: SAME documents/seeds/questions as `03`
(+ `" Output the answer directly."` for retrieval), SAME `temperature=0 /
num_predict=256`, only the context representation changes.


In [ ]:
# Cell 0 — fetch repo (branch Sayok) on Kaggle; safe to re-run
import os, subprocess
if os.path.isdir("/kaggle/working"):
    dest = "/kaggle/working/Testing-dip"
    if os.path.isdir(os.path.join(dest, ".git")):
        subprocess.run(["git", "-C", dest, "pull", "--ff-only"], check=False)
    elif not os.path.isdir(dest):
        subprocess.run(["git", "clone", "-b", "Sayok",
                        "https://github.com/iamahnaf/Testing-dip.git", dest], check=True)
    print("repo:", dest)
else:
    print("not on Kaggle — skipping repo fetch (run from the repo checkout)")


In [ ]:
# Cell 1 — config
OLLAMA_HOST = "https://ollama.com/api"  # cloud; no local server, nothing downloaded
OLLAMA_MODEL = "gemma4:31b"             # dense 31B cloud model (verified via /api/tags)
BUDGETS = [{"tag": "low", "canvas_w": 600}, {"tag": "high", "canvas_w": 1100}]
CONTEXTS = [{"tag": "short", "words": 350}, {"tag": "med", "words": 1500}, {"tag": "long", "words": 3000}]
N_SUMM = 4          # CNN/DailyMail articles (each = 1 text + 1 image call). Lower on quota errors.
SUMM_WORDS = 800    # truncate articles so the rendered image stays legible
GEN = {"temperature": 0.0, "num_predict": 256}
OUT = "/kaggle/working/results/raw/gemma4_ollama_cloud.jsonl"
print(OLLAMA_MODEL, BUDGETS, [c['tag'] for c in CONTEXTS], f"N_SUMM={N_SUMM}")


In [ ]:
# Cell 2 — deps + key + reachability check (1 cheap text-only call)
!pip install -q pillow pandas rouge-score datasets 2>&1 | tail -1
import os, json as _json, urllib.request
key = os.environ.get("OLLAMA_API_KEY", "")
try:
    from kaggle_secrets import UserSecretsClient
    key = key or UserSecretsClient().get_secret("OLLAMA_API_KEY")
except Exception as e:
    print("kaggle_secrets unavailable:", e)
assert key, "Set Kaggle Secret OLLAMA_API_KEY first (ollama.com → Settings → API keys)."

def _post(path, payload, timeout=600):
    req = urllib.request.Request(OLLAMA_HOST + path, data=_json.dumps(payload).encode(),
        headers={"Content-Type": "application/json", "Authorization": "Bearer " + key})
    return _json.load(urllib.request.urlopen(req, timeout=timeout))

r = _post("/api/chat", {"model": OLLAMA_MODEL,
    "messages": [{"role": "user", "content": "Reply with exactly: OK"}],
    "stream": False, "options": {"temperature": 0.0, "num_predict": 8}})
print("reachability:", (r.get("message", {}).get("content", "") or "").strip()[:50],
      "| prompt_eval:", r.get("prompt_eval_count"), "| eval:", r.get("eval_count"))


In [ ]:
# Cell 3 — renderer + samples + metrics + cloud chat helper (PIL standalone; same logic as 01/03)
import base64, io, json, os, random, re, string, textwrap, time, unicodedata
from PIL import Image, ImageDraw, ImageFont

def normalize(t):
    t = unicodedata.normalize("NFKC", t or "")
    return re.sub(r"[ \t]+", " ", t.replace("\r\n", "\n")).strip()

def render(text, width=800, font_size=18, ls=6, wrap=80, pad=30):
    text = normalize(text)
    try: font = ImageFont.truetype("arial.ttf", font_size)
    except IOError: font = ImageFont.load_default()
    lines = []
    for p in text.split("\n\n"):
        lines.extend(textwrap.wrap(p, width=wrap)); lines.append("")
    img = Image.new("RGB", (width, 2*pad + max(len(lines)*(font_size+ls), 100)), "white")
    d = ImageDraw.Draw(img); y = pad
    for ln in lines:
        if ln: d.text((pad, y), ln, fill="black", font=font)
        y += font_size + ls
    return img

def to_b64(img, max_h=2600):
    if img.height > max_h:
        img = img.resize((int(img.width*max_h/img.height), max_h))
    buf = io.BytesIO(); img.convert("RGB").save(buf, "JPEG", quality=72)
    return base64.b64encode(buf.getvalue()).decode(), img.size

DIST = ("The thermal radiator on the secondary boom radiates excess heat into cold space. "
        "Star tracking sensor arrays refresh coordinate quaternions at thirty hertz. ")
def make_sample(words, seed):
    rng = random.Random(seed)
    code = f"CIPHER-{rng.randint(1000, 9999)}-OMEGA"
    needle = f"Special authorization key {code} was recorded in emergency protocol ledger section 9."
    paras, n = [], 0
    while n < words:
        p = " ".join(rng.choice([DIST]) for _ in range(6)); paras.append(p); n += len(p.split())
    w = paras[len(paras)//2].split(); w.insert(len(w)//2, needle)
    paras[len(paras)//2] = " ".join(w)
    return "\n\n".join(paras), code

def norm(s):
    return re.sub(r"\s+", " ", (s or "").lower().translate(str.maketrans("", "", string.punctuation))).strip()
def tok_f1(pred, gt):
    p, g = set(norm(pred).split()), set(norm(gt).split())
    if not p or not g: return 0.0
    tp = len(p & g); prec, rec = tp/len(p), tp/len(g)
    return 0.0 if prec+rec == 0 else 2*prec*rec/(prec+rec)

def ollama_chat(content, images=None, retries=4):
    msg = {"role": "user", "content": content}
    if images: msg["images"] = images
    body = {"model": OLLAMA_MODEL, "messages": [msg], "stream": False,
            "options": {"temperature": GEN["temperature"], "num_predict": GEN["num_predict"]}}
    for a in range(retries):
        try:
            return _post("/api/chat", body)
        except Exception as e:
            if a < retries-1: time.sleep(15*(a+1)); continue
            raise RuntimeError(f"ollama cloud chat failed after retries: {str(e)[:200]}")

def content_of(r):
    return ((r or {}).get("message", {}).get("content", "") or "")

def counters(r):
    r = r or {}
    return {"prompt_eval": r.get("prompt_eval_count"), "eval": r.get("eval_count"),
            "total_s": round(r.get("total_duration", 0)/1e9, 2) if r.get("total_duration") else None}


In [ ]:
# Cell 4 — retrieval A/B loop: text arm vs image arm (9 API calls). Accuracy-only.
import os
from datetime import datetime, timezone
os.makedirs(os.path.dirname(OUT), exist_ok=True)
QSUFFIX = " Output the answer directly."
n = 0
for ci, c in enumerate(CONTEXTS):
    context, gt = make_sample(c["words"], seed=100+ci)  # SAME seeds as 03 → same documents
    question = "What special authorization key was recorded in emergency protocol ledger section 9?" + QSUFFIX
    rt = ollama_chat(f"Context:\n{context}\n\nQuestion: {question}")
    pred_t, ct = content_of(rt), counters(rt)
    for b in BUDGETS:
        img = render(context, width=b["canvas_w"])
        b64, sent_size = to_b64(img)
        ri = ollama_chat(f"Question: {question}", images=[b64])
        pred_i, cii = content_of(ri), counters(ri)
        rec = {"experiment_id": "gemma4_cloud_ret", "timestamp": datetime.now(timezone.utc).isoformat(),
               "model": OLLAMA_MODEL, "runtime": "ollama-cloud-api", "task": "ruler-sniah", "input_mode": "paired-api-accuracy-only",
               "context_tag": c["tag"], "visual_budget_tag": b["tag"], "canvas_w": b["canvas_w"],
               "image_size": f"{img.width}x{img.height}", "image_sent_size": f"{sent_size[0]}x{sent_size[1]}",
               "renderer": "pil-fallback-jpeg72", "context_text_tokens_m": None, "question_tokens_q": None,
               "api_prompt_eval_text": ct["prompt_eval"], "api_prompt_eval_image": cii["prompt_eval"],
               "api_eval_text": ct["eval"], "api_eval_image": cii["eval"],
               "hit_text": int(norm(gt) in norm(pred_t)), "hit_image": int(norm(gt) in norm(pred_i)),
               "f1_text": round(tok_f1(pred_t, gt), 3), "f1_image": round(tok_f1(pred_i, gt), 3),
               "lat_text_s": None, "lat_image_s": None, "vram_text_gb": None, "vram_image_gb": None,
               "temperature": GEN["temperature"], "num_predict": GEN["num_predict"],
               "ground_truth": gt, "pred_text": pred_t.strip()[:300], "pred_image": pred_i.strip()[:300]}
        open(OUT, "a").write(json.dumps(rec) + "\n"); n += 1
        print(f"[{n}/6] {c['tag']} x {b['tag']}: hitT/I={rec['hit_text']}/{rec['hit_image']} f1I={rec['f1_image']}")
print(f"\nRETRIEVAL LOGGED {n} paired rows -> {OUT}")


In [ ]:
# Cell 5 — summarization A/B loop (2*N_SUMM API calls). ROUGE-1/2/L vs reference summary.
from datetime import datetime, timezone
try:
    from datasets import load_dataset
    ds = load_dataset("cnn_dailymail", "3.0.0", split=f"test[:{N_SUMM}]")
    arts = [{"article": r["article"], "ref": r["highlights"]} for r in ds]
    print(f"CNN/DM loaded: {len(arts)} articles")
except Exception as e:
    print("CNN/DM load failed (offline?):", str(e)[:200], "— summarization skipped, retrieval rows already logged.")
    arts = []
from rouge_score import rouge_scorer
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
QSUMM = "Summarize the following document in 3-4 sentences. Output the summary directly."
n2 = 0
for si, a in enumerate(arts):
    doc = " ".join(a["article"].split()[:SUMM_WORDS])
    pred_t = content_of(ollama_chat(f"Document:\n{doc}\n\n{QSUMM}"))
    img = render(doc, width=1100)
    b64, sent_size = to_b64(img)
    pred_i = content_of(ollama_chat(QSUMM, images=[b64]))
    st = scorer.score(a["ref"], pred_t); si_ = scorer.score(a["ref"], pred_i)
    rec = {"experiment_id": "gemma4_cloud_summ", "timestamp": datetime.now(timezone.utc).isoformat(),
           "model": OLLAMA_MODEL, "runtime": "ollama-cloud-api", "task": "cnn-dm-summ", "input_mode": "paired-api-accuracy-only",
           "sample_id": si, "context_text_tokens_m": None,
           "image_size": f"{img.width}x{img.height}", "renderer": "pil-fallback-jpeg72",
           "rouge1_text": round(st["rouge1"].fmeasure, 4), "rouge2_text": round(st["rouge2"].fmeasure, 4),
           "rougeL_text": round(st["rougeL"].fmeasure, 4),
           "rouge1_image": round(si_["rouge1"].fmeasure, 4), "rouge2_image": round(si_["rouge2"].fmeasure, 4),
           "rougeL_image": round(si_["rougeL"].fmeasure, 4),
           "lat_text_s": None, "lat_image_s": None, "pred_text": pred_t.strip()[:500], "pred_image": pred_i.strip()[:500]}
    open(OUT, "a").write(json.dumps(rec) + "\n"); n2 += 1
    print(f"[summ {n2}/{len(arts)}] R-L text={rec['rougeL_text']} image={rec['rougeL_image']}")
print(f"SUMMARIZATION LOGGED {n2} rows -> {OUT}")


In [ ]:
# Cell 6 — aggregate from the JSONL (never hand-edit)
import pandas as pd
df = pd.read_json(OUT, lines=True)
ret = df[df.task == "ruler-sniah"]
if len(ret):
    g = ret.groupby("visual_budget_tag").agg(n=("hit_image", "size"), acc_text=("hit_text", "mean"),
        acc_image=("hit_image", "mean"), f1_image=("f1_image", "mean"))
    print("RETRIEVAL (paired, accuracy-only):"); print(g.round(3).to_string())
summ = df[df.task == "cnn-dm-summ"]
if len(summ):
    print("\nSUMMARIZATION (mean ROUGE F):")
    print(summ[["rouge1_text", "rouge2_text", "rougeL_text", "rouge1_image", "rouge2_image", "rougeL_image"]].mean().round(4).to_string())
print("\nGATE: pass iff 6 retrieval rows logged with hit/f1 non-null. m/q/CR stay null by design (provider usage only).")
